# 12 - Ensemble Methods

## Context
Setiap model gradient boosting (LightGBM, XGBoost, CatBoost) mempelajari pola data dengan cara yang sedikit berbeda. Menggabungkan prediksi dari beberapa model (ensemble) sering kali bisa menghasilkan prediksi yang lebih stabil, mengurangi varians, dan meningkatkan performa secara keseluruhan.

## Objective
- Menggabungkan prediksi model menggunakan metode Hard Voting, Soft Voting, dan Weighted Soft Voting.
- Melatih model Stacking (Meta-Learner) menggunakan regresi logistik di atas prediksi base models.
- Mengevaluasi dan membandingkan performa setiap metode ensemble.

### Impor Library

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

### Muat Dataset Terproses & Pemisahan Train/Val

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)

exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

### Pelatihan Ulang Model (Base Models)

In [ ]:
# Melatih model LightGBM base
dtrain_lgb = lgb.Dataset(X_tr, label=y_tr)
dval_lgb = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb)
model_lgb = lgb.train(
    {"objective": "binary", "metric": "auc", "num_leaves": 31, "learning_rate": 0.1, "verbose": -1, "random_state": 42},
    dtrain_lgb, num_boost_round=100, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(15, verbose=False)]
)

# Melatih model XGBoost base
dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr)
dval_xgb = xgb.DMatrix(X_va, label=y_va)
model_xgb = xgb.train(
    {"objective": "binary:logistic", "eval_metric": "auc", "max_depth": 6, "learning_rate": 0.1, "verbosity": 0, "random_state": 42},
    dtrain_xgb, num_boost_round=100, evals=[(dval_xgb, "val")],
    callbacks=[xgb.callback.EarlyStopping(rounds=15, save_best=True, metric_name="auc", data_name="val")],
    verbose_eval=False
)

# Melatih model CatBoost base
model_cb = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, eval_metric="AUC", random_seed=42, verbose=False)
model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=15)

print("Models retrained and loaded.")

### Prediksi Model Individu

In [ ]:
preds_lgb = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
preds_xgb = model_xgb.predict(dval_xgb)
preds_cb = model_cb.predict_proba(X_va)[:, 1]

auc_lgb = roc_auc_score(y_va, preds_lgb)
auc_xgb = roc_auc_score(y_va, preds_xgb)
auc_cb = roc_auc_score(y_va, preds_cb)

individual_scores = {"lightgbm": auc_lgb, "xgboost": auc_xgb, "catboost": auc_cb}
print(pd.DataFrame([{"Model": k, "Val AUC": v} for k, v in individual_scores.items()]))

### Ensemble: Hard Voting

In [ ]:
# Hard voting: mengambil keputusan berdasarkan suara mayoritas (prediksi kelas >= 0.5)
hard_lgb = (preds_lgb >= 0.5).astype(int)
hard_xgb = (preds_xgb >= 0.5).astype(int)
hard_cb = (preds_cb >= 0.5).astype(int)

votes = hard_lgb + hard_xgb + hard_cb
preds_hard = (votes >= 2).astype(int)

f1_hard = f1_score(y_va, preds_hard)
prec_hard = precision_score(y_va, preds_hard)
rec_hard = recall_score(y_va, preds_hard)
print(f"Hard Voting \u2014 F1: {f1_hard:.4f}, Precision: {prec_hard:.4f}, Recall: {rec_hard:.4f}")

### Ensemble: Soft Voting

In [ ]:
# Soft voting: rata-rata nilai probabilitas dari ketiga model
preds_soft = (preds_lgb + preds_xgb + preds_cb) / 3.0
auc_soft = roc_auc_score(y_va, preds_soft)
preds_soft_class = (preds_soft >= 0.5).astype(int)

f1_soft = f1_score(y_va, preds_soft_class)
prec_soft = precision_score(y_va, preds_soft_class)
rec_soft = recall_score(y_va, preds_soft_class)

print(f"Soft Voting \u2014 Val AUC: {auc_soft:.5f}")
print(f"  F1: {f1_soft:.4f}, Precision: {prec_soft:.4f}, Recall: {rec_soft:.4f}")

### Ensemble: Weighted Soft Voting

In [ ]:
# Weighted soft voting: bobot model ditentukan proporsional berdasarkan nilai AUC-nya
w_lgb, w_xgb, w_cb = auc_lgb, auc_xgb, auc_cb
w_sum = w_lgb + w_xgb + w_cb
preds_wsoft = (w_lgb * preds_lgb + w_xgb * preds_xgb + w_cb * preds_cb) / w_sum
auc_wsoft = roc_auc_score(y_va, preds_wsoft)
print(f"Weighted Soft Voting \u2014 Val AUC: {auc_wsoft:.5f}")

### Ensemble: Stacking (Meta-Learner)

In [ ]:
# Gunakan Logistic Regression sebagai meta-classifier di atas prediksi model base
X_meta_val = pd.DataFrame({
    "lgb": preds_lgb,
    "xgb": preds_xgb,
    "cb": preds_cb
})

# Bagi prediksi validasi menjadi train/val internal untuk simulasi out-of-fold
X_tr_meta, X_va_meta, y_tr_meta, y_va_meta = train_test_split(
    X_meta_val, y_va, test_size=0.5, stratify=y_va, random_state=42
)

meta_model = LogisticRegression(class_weight="balanced", random_state=42)
meta_model.fit(X_tr_meta, y_tr_meta)

preds_stack = meta_model.predict_proba(X_va_meta)[:, 1]
auc_stack = roc_auc_score(y_va_meta, preds_stack)
preds_stack_class = (preds_stack >= 0.5).astype(int)

f1_stack = f1_score(y_va_meta, preds_stack_class)
prec_stack = precision_score(y_va_meta, preds_stack_class)
rec_stack = recall_score(y_va_meta, preds_stack_class)

print(f"Stacking (Meta-Learner) \u2014 Val AUC: {auc_stack:.5f}")
print(f"  F1: {f1_stack:.4f}, Precision: {prec_stack:.4f}, Recall: {rec_stack:.4f}")

### Perbandingan Semua Metode Ensemble

In [ ]:
comparison = pd.DataFrame([
    {"Method": "LightGBM", "Val AUC": auc_lgb},
    {"Method": "XGBoost", "Val AUC": auc_xgb},
    {"Method": "CatBoost", "Val AUC": auc_cb},
    {"Method": "Soft Voting", "Val AUC": auc_soft},
    {"Method": "Weighted Soft", "Val AUC": auc_wsoft},
    {"Method": "Stacking", "Val AUC": auc_stack},
])
comparison = comparison.sort_values("Val AUC", ascending=False).reset_index(drop=True)
print(comparison)

### Visualisasi Perbandingan Model Ensemble

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(comparison["Method"], comparison["Val AUC"], color="#4c72b0")
ax.set_ylabel("Validation AUC")
ax.set_title("Ensemble Methods \u2014 Comparison")
ax.set_ylim(0.85, 1.0)
for bar, val in zip(bars, comparison["Val AUC"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:.4f}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
Path("../data/metadata").mkdir(parents=True, exist_ok=True)
plt.savefig("../data/metadata/ensemble_comparison.png")
plt.show()